![](https://github.com/destination-earth/DestinE-DataLake-Lab/blob/main/img/DestinE-banner.jpg?raw=true)


# DEDL - HDA Tutorial - access HydroLand

**Author**: EUMETSAT / UFZ <br>
**Copyright**: 2026 EUMETSAT <br>
**Licence**: MIT <br>
**Modified by**: jeilealr

<div class="alert alert-block alert-warning">
<b> Prerequisites: </b>
<li> For Data access : <a href="https://platform.destine.eu/"> DestinE user account and access to restricted datasets</a> </li>
<li> Helper modules <code>hda_http.py</code>, <code>hda_stac.py</code>, <code>hda_download.py</code> live in <code>example_tools/</code> at the repository root.</li>
</div>

### Import the relevant modules

In [ ]:
import sys
from pathlib import Path

# Add example_tools/ to the module search path (one level up from example_apps/)
sys.path.insert(0, str(Path("..") / "example_tools"))

In [ ]:
import glob
import json
from getpass import getpass

import xarray as xr
import destinelab as deauth

from hda_http import request_json
from hda_stac import extract_all_items
from hda_download import download_item_archive, download_item_assets

### Define some constants for the API URL


In [ ]:
# Define the collection to be used
COLLECTION_ID = "EO.UFZ.DAT.HYDROLAND"

# Core API
HDA_API_URL = "https://hda.data.destination-earth.eu"

# STAC API
STAC_API_URL = f"{HDA_API_URL}/stac/v2"
COLLECTIONS_URL = f"{STAC_API_URL}/collections"
COLLECTION_BY_ID_URL = f"{COLLECTIONS_URL}/{COLLECTION_ID}"
COLLECTION_ITEMS_URL = f"{COLLECTIONS_URL}/{COLLECTION_ID}/items"
SEARCH_URL = f"{STAC_API_URL}/search"

# Request settings
REQUEST_TIMEOUT = 30    # seconds
REQUEST_RETRIES = 3
DOWNLOAD_CHUNK_SIZE = 1_048_576  # 1 MB

## Authenticate

In [ ]:
DESP_USERNAME = input("Please input your DESP username or email: ")
DESP_PASSWORD = getpass("Please input your DESP password: ")

auth = deauth.AuthHandler(DESP_USERNAME, DESP_PASSWORD)
access_token = auth.get_token()
if access_token is not None:
    print("DEDL/DESP Access Token Obtained Successfully")
else:
    print("Failed to Obtain DEDL/DESP Access Token")

auth_headers = {"Authorization": f"Bearer {access_token}"}

### Discover data - Authenticated

Once authenticated, we can discover the collection.

In [ ]:
collection_metadata = request_json(
    "GET",
    COLLECTION_BY_ID_URL,
    headers=auth_headers,
    timeout=REQUEST_TIMEOUT,
    retries=REQUEST_RETRIES,
)
print(json.dumps(collection_metadata, indent=4))

## Search

Once a collection is selected, you can search for items that match the specified input filters and order the results.

Adjust `SEARCH_DATETIME`, `SEARCH_BBOX`, and `SEARCH_LIMIT` to narrow the results. The collection currrently covers IFS-NEMO **1990-01-01 to 2049-12-31**.

In [ ]:
SEARCH_DATETIME = "1995-01-01T00:00:00Z/1995-12-31T00:00:00Z" # Select the wanted range
SEARCH_BBOX = [-180, -90, 180, 90]
SEARCH_LIMIT = 1
SEARCH_SORTBY = [{"field": "datetime", "direction": "desc"}]

BODY = {
    "collections": [COLLECTION_ID],
    "datetime": SEARCH_DATETIME,
    "bbox": SEARCH_BBOX,
    "sortby": SEARCH_SORTBY,
    "limit": SEARCH_LIMIT,
}

search_json = request_json(
    "POST",
    SEARCH_URL,
    headers=auth_headers,
    json_body=BODY,
    timeout=REQUEST_TIMEOUT,
    retries=REQUEST_RETRIES,
)
print(json.dumps(search_json, indent=4))

## Download - individual assets (recommended)

Downloads each `.nc` file individually from S3 signed URLs — faster than the zip archive approach below.

In [ ]:
download_item_assets(
    search_json=search_json,
    collection_id=COLLECTION_ID,
    auth_headers=auth_headers,
    request_timeout=REQUEST_TIMEOUT,
    chunk_size=DOWNLOAD_CHUNK_SIZE,
    existing_policy="skip",
)

## Download - item archive via downloadLink (optional)

Downloads the complete item as a `.zip` archive. Use this if you want everything in a single file.

In [ ]:
items = extract_all_items(search_json)

for item in items:
    download_url = item.get("assets", {}).get("downloadLink", {}).get("href")
    if not download_url:
        raise ValueError("Search result does not include assets.downloadLink.href")

    item_id = item.get("id")
    if not item_id:
        raise ValueError("Search result does not include item id")

    print("Download URL:", download_url)
    print("Item ID:", item_id)

    filename = download_item_archive(
        download_url=download_url,
        item_id=item_id,
        headers=auth_headers,
        timeout=REQUEST_TIMEOUT,
        chunk_size=DOWNLOAD_CHUNK_SIZE,
        existing_policy="skip",
    )
    print(f"Download successful! File saved as: {filename}")

# Display the downloaded file

The downloaded NetCDF files can be opened directly with xarray.

In [ ]:
nc_files = glob.glob(f"{COLLECTION_ID}/**/*.nc", recursive=True)
if nc_files:
    local_file = nc_files[0]
    print(f"Opening: {local_file}")
    ds = xr.open_dataset(local_file)
    print(ds)
else:
    print("No .nc files found. Run the download cells above first.")